In [ ]:
# =========================
# nb_summary
# =========================
# Phase 9, SC-17: Generate cross-DFM run summary (replaces/augments nb_reports final step)
# Provides overall narrative across all DFMs for the run period

# ---------- Parameters ----------
period = "2026-03"
run_id = "manual_test_run"

# ---------- Imports ----------
from pyspark.sql import functions as F
from datetime import datetime
import sys
import json

# ---------- Pre-flight checks ----------
try:
    mssparkutils
except NameError:
    print("[ERROR] mssparkutils not available - not running in Fabric context")
    sys.exit(1)

try:
    if not period or not period.strip():
        raise ValueError("period parameter is empty")
    if not run_id or not run_id.strip():
        raise ValueError("run_id parameter is empty")
    print(f"[nb_summary] START period={period}, run_id={run_id}")
except ValueError as e:
    print(f"[ERROR] Invalid parameters: {e}")
    sys.exit(1)

# Verify tables
if not spark.catalog.tableExists("canonical_holdings") or not spark.catalog.tableExists("run_summary"):
    print("[ERROR] Required tables do not exist")
    sys.exit(1)

# Load shared_ai_utils
sys.path.append('/lakehouse/default/Files/config')
try:
    from shared_ai_utils import load_ai_config
    ai_enabled = True
except ImportError as e:
    print(f"[WARNING] shared_ai_utils not found: {e}")
    print("[nb_summary] AI not available; skipping")
    mssparkutils.notebook.exit("AI_DISABLED")

# Load AI config
try:
    ai_client = load_ai_config('/lakehouse/default/Files/config/azure_openai_config.json')
    if ai_client is None:
        raise RuntimeError("AI config returned None")
except Exception as e:
    print(f"[ERROR] Failed to load AI config: {type(e).__name__}: {e}")
    mssparkutils.notebook.exit("CONFIG_ERROR")

# ---------- Aggregate run statistics ----------
try:
    ch = spark.table("canonical_holdings").filter(
        (F.col("period") == period) & (F.col("run_id") == run_id)
    )
    
    if ch.rdd.isEmpty():
        print(f"[nb_summary] No data for period={period}, run_id={run_id}")
        mssparkutils.notebook.exit("NO_DATA")
    
    # Overall counts
    pa = spark.table("policy_aggregates").filter(
        (F.col("period") == period) & (F.col("run_id") == run_id)
    )
    
    total_policies = pa.count()
    total_holdings = ch.count()
    
    total_value_result = pa.agg(F.sum("total_bid_value_gbp")).collect()[0][0]
    total_value = float(total_value_result or 0)
    
    # Exception summary
    ve = spark.table("validation_events").filter(
        (F.col("period") == period) & (F.col("run_id") == run_id)
    )
    
    total_exceptions = ve.filter(F.col("status") == "fail").count()
    exception_summary = ve.filter(F.col("status") == "fail").groupBy("severity").agg(F.count("*")).collect()
    exception_dict = {str(r["severity"]): int(r[1]) for r in exception_summary}
    
    # DFM-level breakdown
    dfm_summary = ch.groupBy("dfm_id").agg(
        F.count("*").alias("holdings"),
        F.countDistinct("policy_id").alias("policies")
    ).collect()
    
    dfm_list = [
        {
            "dfm": str(r["dfm_id"]) if r["dfm_id"] else "UNKNOWN",
            "policies": int(r["policies"]),
            "holdings": int(r["holdings"])
        }
        for r in dfm_summary
    ]
    
    print(f"[nb_summary] Aggregated data: {total_policies} policies, {total_holdings} holdings, {len(dfm_list)} DFMs")

except Exception as e:
    print(f"[ERROR] Failed to aggregate statistics: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)

# ---------- Generate summary narrative ----------
try:
    prompt = f"""
Provide an executive summary for the portfolio reconciliation run:
- Period: {period}
- Run ID: {run_id}
- Total policies: {total_policies}
- Total holdings: {total_holdings}
- Total portfolio value (GBP): {total_value:,.2f}
- Total validation exceptions: {total_exceptions}
- Exception breakdown by severity: {json.dumps(exception_dict)}
- DFM breakdown: {json.dumps(dfm_list)}

Provide:
1. Executive summary (2-3 sentences)
2. Overall health assessment
3. Top risks or issues (if any)
4. Overall recommendation for the period

Return JSON: {{
  "summary": "...",
  "health": "...",
  "risks": [...],
  "recommendation": "..."
}}
"""
    
    try:
        result_text = ai_client.generate_narrative(prompt)
        
        if not result_text or not isinstance(result_text, str):
            raise ValueError("AI returned non-string response")
        
        result = json.loads(result_text)
        summary_narrative = str(result.get("summary", ""))
        health_assessment = str(result.get("health", ""))
        recommendation = str(result.get("recommendation", ""))
    
    except json.JSONDecodeError as e:
        print(f"[WARNING] AI response not valid JSON: {e}")
        summary_narrative = (
            f"Period {period}: {total_policies} policies, {total_holdings} holdings, "
            f"GBP {total_value:,.2f}; {total_exceptions} validation exceptions"
        )
        health_assessment = "Unable to assess (AI JSON parse error)"
        recommendation = "Manual review recommended"
    
    except Exception as e:
        print(f"[WARNING] AI generation failed: {type(e).__name__}: {e}")
        summary_narrative = (
            f"Period {period}: {total_policies} policies, {total_holdings} holdings, "
            f"GBP {total_value:,.2f}; {total_exceptions} validation exceptions"
        )
        health_assessment = "Unable to assess (AI unavailable)"
        recommendation = "Manual review recommended"

    # Write run_summary table
    try:
        spark.sql(f"""
            INSERT INTO run_summary
            (period, run_id, summary_text, run_start_time, run_end_time, total_policies, total_holdings, validation_exceptions_count, exception_summary_json, generated_at)
            VALUES
            ('{period}', '{run_id}', '{summary_narrative.replace("'", "''")}', current_timestamp(), current_timestamp(), {total_policies}, {total_holdings}, {total_exceptions}, '{json.dumps(exception_dict).replace("'", "''")}', current_timestamp())
        """)
        
        print(f"[nb_summary] COMPLETE - run summary stored")
        print(f"  Period: {period}")
        print(f"  Total policies: {total_policies}")
        print(f"  Total holdings: {total_holdings}")
        print(f"  Total value (GBP): {total_value:,.2f}")
        print(f"  Validation exceptions: {total_exceptions}")
        print(f"  Health: {health_assessment}")
        print(f"  Recommendation: {recommendation}")
        
        mssparkutils.notebook.exit("OK")
    
    except Exception as e:
        print(f"[ERROR] Failed to write run_summary: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)

except Exception as e:
    print(f"[ERROR] Summary generation failed: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)